# pa2-sft — Task 2 baseline generation + Task 3 SFT
Settings: GPU T4 x2, Internet on. Launch with **Save Version → Save & Run All (Commit)**. ~1.5–2 h.
SFT trains on cuda:0 while base-model generations run on cuda:1 in the background.

In [ ]:
# ---- Setup: identical first cell in every notebook (a commit starts on a fresh machine) ----
GH_USER, REPO = "mohdddd", "small-model-alignment-starter"
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/{GH_USER}/{REPO}.git repo
%cd /kaggle/working/repo
!git log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
import os, sys, json, glob, shutil, subprocess
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("outputs", exist_ok=True)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/ 2>&1 | tail -3

In [ ]:
!python scripts/prepare_data.py --which sft alpaca --out_dir outputs/data
print(open('outputs/data/data_stats.json').read())

In [ ]:
# Task 2: base-model generations on cuda:1 (background) while SFT uses cuda:0
os.makedirs("outputs/generations", exist_ok=True)
BASE_CMD = (
    "python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set outputs/data/alpaca_eval_300_seed42.json "
    "--out outputs/generations/alpaca_base.json --generator qwen2.5-0.5b-base --device cuda:1 && "
    "python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set eval_sets/hh_rlhf_eval_300_seed42.json "
    "--out outputs/generations/hh_base.json --generator qwen2.5-0.5b-base --device cuda:1")
base_log = open("outputs/base_generation.log", "w")
base_proc = subprocess.Popen(BASE_CMD, shell=True, stdout=base_log, stderr=subprocess.STDOUT)
print("base generation started, pid", base_proc.pid)

In [ ]:
!python scripts/train_sft.py --train_file outputs/data/sft_train.jsonl --val_file outputs/data/sft_val.jsonl --out_dir outputs/sft --device cuda:0

In [ ]:
rc = base_proc.wait(); base_log.close()
print("base generation exit code:", rc)
print(open("outputs/base_generation.log").read()[-2500:])

In [ ]:
from IPython.display import Image, display
def show(path):
    display(Image(path))
!python scripts/plot_metrics.py --run_dir outputs/sft --kind sft
show('outputs/sft/sft_curves.png')
print(open('outputs/sft/summary.json').read())

Sanity check: SFT best checkpoint on 3 AlpacaEval prompts (checks path, tokenizer, template, EOS, length).

In [ ]:
!python scripts/generate.py --model outputs/sft/best --eval_set outputs/data/alpaca_eval_300_seed42.json --out outputs/generations/alpaca_sft.json --generator qwen2.5-0.5b-sft --limit 3 --device cuda:1

In [ ]:
!cd outputs && zip -qr /kaggle/working/sft_run_outputs.zip sft/metrics.jsonl sft/config.json sft/summary.json sft/sft_curves.png sft/best/checkpoint_meta.json data/data_stats.json data/alpaca_eval_300_seed42.json generations base_generation.log
!du -sh /kaggle/working/repo/outputs/* 2>/dev/null; df -h /kaggle/working | tail -1